In [ ]:
# ===== HEURISTIC SEARCH — CONFIG (edit ONLY this cell) =====================
# Runs the tuned knot ordering against the length baseline on the SAME
# presentations, at a Colab-scale budget. Both arms in one run, so the
# comparison is like-for-like.

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "research/w5/stable-ac-escape"
CLONE      = True
UPDATE_REPO = True          # Restart -> Run All picks up pushed .py fixes

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch"   # NEVER /content/drive alone

cfg = dict(
    # ---- what to run --------------------------------------------------
    # "bench66"    the 66-row benchmark (60 graded + 6 open) — start here
    # "unsolved124" the 124 unsolved AC-classes — the real target
    DATASET   = "bench66",
    SUBSET    = None,        # None = all rows; or an int to take the first N

    # ---- the two arms -------------------------------------------------
    # Same function, same returned dict; only the ordering differs.
    ARMS      = ["baseline", "recommended"],

    # ---- budget -------------------------------------------------------
    # THE measurement that matters: checkpoints let you see whether the
    # tuned arm's ADVANTAGE keeps widening past 1,000 nodes. If it flattens,
    # the extrapolation from the local study does not hold — see the README.
    NODE_BUDGET = 100_000,
    CHECKPOINTS = [500, 1_000, 2_000, 5_000, 10_000, 25_000, 50_000, 100_000],

    # ---- the search ---------------------------------------------------
    # 48, not 24: the climb pops relators past 30 and capping truncates it.
    MAX_RELATOR_LENGTH = 48,

    # ---- output -------------------------------------------------------
    RESUME    = True,        # skip (arm, presentation) rows already on disk
    OUT_STEM  = "hsearch_ab",
)

# Heartbeat: TIME-based, never event-based — a slow CPU must show as a
# falling rate, not as silence.
HEARTBEAT_SECS = 60          # in-search beat with instantaneous nodes/s
PROGRESS_SECS  = 300         # cumulative done/solved/ETA line


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor: re-runs must not nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)   # under MyDrive, never the mount root
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# A `git reset --hard` rewrites the .py files, but Python keeps the OLD module
# objects in sys.modules for the life of the runtime — a pull is NOT a reload.
# Drop them so the next import reads what SETUP just fetched.
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# Warm the numba kernels once, so the first timed search is not paying for JIT.
from experiments.heuristic_search.hsolve import greedy_search_h, RECOMMENDED
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32, config=RECOMMENDED)
print("kernels warm — setup done")


In [ ]:
# ==================== RUN =================================================
from experiments.heuristic_search.run_ab import run_ab
run_ab(cfg, out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else "results/hsearch"),
       heartbeat_secs=HEARTBEAT_SECS, progress_secs=PROGRESS_SECS)
